In [4]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Real-Time 3D Motion Model</title>
<style>
  html, body {
    margin: 0;
    width: 100%;
    height: 100%;
    overflow: hidden;
    background: #02050a;
  }
  svg {
    width: 100vw;
    height: 100vh;
    display: block;
  }
</style>
</head>
<body>
<svg xmlns="http://www.w3.org/2000/svg" width="1400" height="1100" viewBox="0 0 1400 1100">
  <defs>
    <radialGradient id="bgGlow" cx="50%" cy="12%" r="60%">
      <stop offset="0%" stop-color="#12335f"/>
      <stop offset="48%" stop-color="#07111d"/>
      <stop offset="100%" stop-color="#02050a"/>
    </radialGradient>

    <filter id="blueGlow" x="-50%" y="-50%" width="200%" height="200%">
      <feGaussianBlur stdDeviation="3.5" result="blur"/>
      <feMerge>
        <feMergeNode in="blur"/>
        <feMergeNode in="SourceGraphic"/>
      </feMerge>
    </filter>

    <filter id="cyanGlow" x="-50%" y="-50%" width="200%" height="200%">
      <feGaussianBlur stdDeviation="2.5" result="blur"/>
      <feMerge>
        <feMergeNode in="blur"/>
        <feMergeNode in="SourceGraphic"/>
      </feMerge>
    </filter>

    <linearGradient id="signalGrad" x1="0" x2="1">
      <stop offset="0%" stop-color="#0f63ff"/>
      <stop offset="50%" stop-color="#36a8ff"/>
      <stop offset="100%" stop-color="#0f63ff"/>
    </linearGradient>

    <style>
      .grid { stroke: rgba(210,225,245,0.18); stroke-width: 1; fill: none; }
      .gridStrong { stroke: rgba(220,235,255,0.36); stroke-width: 1.4; fill: none; }
      .edge { stroke: rgba(220,235,255,0.55); stroke-width: 1.6; fill: none; }
      .label { fill: rgba(245,248,252,0.96); font-family: Arial, Helvetica, sans-serif; font-size: 24px; }
      .tick { fill: rgba(245,248,252,0.94); font-family: Arial, Helvetica, sans-serif; font-size: 18px; }
      .legendText { fill: white; font-family: Arial, Helvetica, sans-serif; font-size: 28px; font-weight: 600; }
      .signal { fill: none; stroke: url(#signalGrad); stroke-width: 4; stroke-linejoin: round; stroke-linecap: round; filter: url(#blueGlow); }
      .reflection { fill: none; stroke: #18d6ff; stroke-width: 2.4; stroke-dasharray: 12 10; stroke-linejoin: round; stroke-linecap: round; filter: url(#cyanGlow); }
      .trail { fill: none; stroke: #0f63ff; stroke-width: 2; opacity: .22; filter: url(#blueGlow); }
    </style>
  </defs>

  <rect width="1400" height="1100" fill="url(#bgGlow)"/>

  <!-- Title -->
  <text x="700" y="78" class="label" text-anchor="middle" font-size="34">demo</text>

  <!-- Back and floor grid, pre-projected to avoid JavaScript -->
  <g id="grid">
    <!-- Vertical amplitude grid on rear wall -->
    <path class="gridStrong" d="M425 270 L1220 370"/>
    <path class="grid" d="M392 320 L1190 420"/>
    <path class="grid" d="M360 370 L1160 470"/>
    <path class="gridStrong" d="M328 420 L1130 520"/>
    <path class="grid" d="M296 470 L1100 570"/>
    <path class="grid" d="M264 520 L1070 620"/>
    <path class="gridStrong" d="M232 570 L1040 670"/>
    <path class="grid" d="M200 620 L1010 720"/>
    <path class="grid" d="M168 670 L980 770"/>

    <!-- Floor time grid -->
    <path class="gridStrong" d="M165 820 L1010 965"/>
    <path class="grid" d="M250 770 L1090 915"/>
    <path class="gridStrong" d="M335 720 L1170 865"/>
    <path class="grid" d="M420 670 L1250 815"/>
    <path class="gridStrong" d="M505 620 L1330 765"/>

    <!-- Series direction grid -->
    <path class="gridStrong" d="M165 820 L505 620"/>
    <path class="grid" d="M335 850 L675 650"/>
    <path class="grid" d="M505 880 L845 680"/>
    <path class="grid" d="M675 910 L1015 710"/>
    <path class="gridStrong" d="M845 940 L1185 740"/>
    <path class="gridStrong" d="M1010 965 L1330 765"/>

    <!-- Box edges -->
    <path class="edge" d="M165 820 L1010 965 L1330 765 L505 620 Z"/>
    <path class="edge" d="M165 820 L425 270 L1220 370 L1330 765"/>
    <path class="edge" d="M425 270 L505 620"/>
    <path class="edge" d="M1220 370 L1010 965"/>
  </g>

  <!-- Labels -->
  <text x="555" y="1030" class="label" text-anchor="middle">Time Step</text>
  <text x="78" y="560" class="label" text-anchor="middle" transform="rotate(-90 78 560)">Amplitude</text>
  <text x="1295" y="900" class="label" text-anchor="middle" transform="rotate(-55 1295 900)">Series</text>

  <text x="170" y="850" class="tick" text-anchor="middle">0</text>
  <text x="335" y="880" class="tick" text-anchor="middle">200</text>
  <text x="505" y="910" class="tick" text-anchor="middle">400</text>
  <text x="675" y="940" class="tick" text-anchor="middle">600</text>
  <text x="845" y="970" class="tick" text-anchor="middle">800</text>
  <text x="1015" y="995" class="tick" text-anchor="middle">1000</text>

  <text x="1260" y="780" class="label">Reflection</text>
  <text x="1165" y="870" class="label">Signal</text>

  <text x="128" y="322" class="tick" text-anchor="end">1.50</text>
  <text x="128" y="422" class="tick" text-anchor="end">1.00</text>
  <text x="128" y="522" class="tick" text-anchor="end">0.50</text>
  <text x="128" y="622" class="tick" text-anchor="end">0.00</text>
  <text x="128" y="722" class="tick" text-anchor="end">-0.50</text>
  <text x="128" y="810" class="tick" text-anchor="end">-0.75</text>

  <!-- Legend -->
  <g transform="translate(1110 180)">
    <rect width="225" height="92" rx="10" fill="rgba(0,0,0,.35)" stroke="rgba(255,255,255,.30)"/>
    <line x1="25" y1="32" x2="85" y2="32" stroke="#2f85ff" stroke-width="4" filter="url(#blueGlow)"/>
    <text x="110" y="40" class="legendText">Signal</text>
    <line x1="25" y1="67" x2="85" y2="67" stroke="#18d6ff" stroke-width="3" stroke-dasharray="12 8" filter="url(#cyanGlow)"/>
    <text x="110" y="76" class="legendText">Reflection</text>
  </g>

  <!-- Reflection paths -->
  <g>
    <path class="reflection" d="M275 675 C390 666 515 660 650 652 S920 640 1085 634"/>
    <path class="reflection" d="M350 765 C480 758 610 752 745 745 S990 735 1185 728"/>
    <animateTransform attributeName="transform" type="translate"
      values="0 0; -22 4; 0 0; 22 -4; 0 0" dur="3.5s" repeatCount="indefinite"/>
  </g>

  <!-- Signal trails -->
  <g opacity=".55">
    <path class="trail" d="M185 735 L195 690 L205 765 L215 700 L225 760 L235 680 L245 730 L255 640 L265 755 L275 710
      C310 690 330 640 350 560
      C390 400 450 380 510 410
      L520 340 L530 455 L540 315 L550 470 L560 360 L570 425 L580 300 L590 465 L600 350 L610 440
      L620 335 L630 470 L640 360 L650 430 L660 310 L670 470 L680 345 L690 425 L700 335 L710 470 L720 350 L730 430
      C775 425 815 415 850 430
      C885 450 915 520 930 600
      L940 660 L950 585 L960 720 L970 600 L980 745 L990 620 L1000 705 L1010 640 L1020 760 L1030 620
      L1040 720 L1050 635 L1060 760 L1070 650 L1080 720 L1090 635 L1100 760 L1110 650 L1120 710"/>
    <animateTransform attributeName="transform" type="translate"
      values="-8 4; 12 -6; -8 4" dur="1.2s" repeatCount="indefinite"/>
  </g>

  <!-- Main signal path animated with path shape changes -->
  <path class="signal">
    <animate attributeName="d" dur="1.4s" repeatCount="indefinite"
      values="
      M185 735 L195 690 L205 765 L215 700 L225 760 L235 680 L245 730 L255 640 L265 755 L275 710
      C310 690 330 640 350 560
      C390 400 450 380 510 410
      L520 340 L530 455 L540 315 L550 470 L560 360 L570 425 L580 300 L590 465 L600 350 L610 440
      L620 335 L630 470 L640 360 L650 430 L660 310 L670 470 L680 345 L690 425 L700 335 L710 470 L720 350 L730 430
      C775 425 815 415 850 430
      C885 450 915 520 930 600
      L940 660 L950 585 L960 720 L970 600 L980 745 L990 620 L1000 705 L1010 640 L1020 760 L1030 620
      L1040 720 L1050 635 L1060 760 L1070 650 L1080 720 L1090 635 L1100 760 L1110 650 L1120 710;

      M185 715 L195 770 L205 680 L215 760 L225 690 L235 740 L245 660 L255 750 L265 680 L275 745
      C310 670 330 610 350 540
      C390 420 450 360 510 395
      L520 455 L530 325 L540 470 L550 350 L560 435 L570 310 L580 465 L590 345 L600 450 L610 335
      L620 470 L630 355 L640 430 L650 315 L660 455 L670 350 L680 475 L690 330 L700 450 L710 360 L720 465 L730 340
      C775 450 815 395 850 440
      C885 470 915 540 930 620
      L940 580 L950 735 L960 610 L970 765 L980 630 L990 720 L1000 610 L1010 745 L1020 635 L1030 760
      L1040 640 L1050 725 L1060 615 L1070 745 L1080 630 L1090 760 L1100 640 L1110 730 L1120 620;

      M185 735 L195 690 L205 765 L215 700 L225 760 L235 680 L245 730 L255 640 L265 755 L275 710
      C310 690 330 640 350 560
      C390 400 450 380 510 410
      L520 340 L530 455 L540 315 L550 470 L560 360 L570 425 L580 300 L590 465 L600 350 L610 440
      L620 335 L630 470 L640 360 L650 430 L660 310 L670 470 L680 345 L690 425 L700 335 L710 470 L720 350 L730 430
      C775 425 815 415 850 430
      C885 450 915 520 930 600
      L940 660 L950 585 L960 720 L970 600 L980 745 L990 620 L1000 705 L1010 640 L1020 760 L1030 620
      L1040 720 L1050 635 L1060 760 L1070 650 L1080 720 L1090 635 L1100 760 L1110 650 L1120 710"/>
  </path>

  <!-- Moving scan dots -->
  <g filter="url(#blueGlow)">
    <circle r="7" fill="white">
      <animateMotion dur="6s" repeatCount="indefinite"
        path="M185 735 L195 690 L205 765 L215 700 L225 760 L235 680 L245 730 L255 640 L265 755 L275 710
        C310 690 330 640 350 560 C390 400 450 380 510 410
        L520 340 L530 455 L540 315 L550 470 L560 360 L570 425 L580 300 L590 465 L600 350 L610 440
        L620 335 L630 470 L640 360 L650 430 L660 310 L670 470 L680 345 L690 425 L700 335 L710 470 L720 350 L730 430
        C775 425 815 415 850 430 C885 450 915 520 930 600
        L940 660 L950 585 L960 720 L970 600 L980 745 L990 620 L1000 705 L1010 640 L1020 760 L1030 620
        L1040 720 L1050 635 L1060 760 L1070 650 L1080 720 L1090 635 L1100 760 L1110 650 L1120 710"/>
    </circle>
  </g>

  <g filter="url(#cyanGlow)">
    <circle r="5" fill="white">
      <animateMotion dur="5s" repeatCount="indefinite"
        path="M350 765 C480 758 610 752 745 745 S990 735 1185 728"/>
    </circle>
  </g>
</svg>
</body>
</html>